# Week 7 — Monday: Decision Trees

**DATA 202 · Calvin University**

Autism is diagnosed by observing a child's behavior, often not until the child is several years old. A blood test could, in principle, flag it earlier.

A 2017 study took 24 blood measurements from children aged 3–10: some diagnosed with autism, some not.

> 🎯 **Can a few yes/no questions about a blood sample tell the two groups apart?**

**Today's plan (~45 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · Look at the data |
| ~10 min | 2 · One question: a tree with one split (SLO 07A) |
| ~7 min | 3 · Two more questions: depth 2 |
| ~10 min | 4 · Too many questions: overfitting (SLO 07A) |
| ~10 min | 5 · Keeping trees small: pruning |
| ~3 min | 6 · A rule for sorting children |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk

<sub>Data: Howsmon, Kruger, Melnyk, James & Hahn (2017), *PLOS Computational Biology* 13(3): e1005385, CC BY 4.0 — from the Arkansas Children's Hospital autism IMAGE study</sub>

---
## 1 · Look at the Data · ~5 min

**autism** — one row per child:

* `Group` — **ASD** (diagnosed with autism), **NEU** (neurotypical), **SIB** (sibling of an ASD child, not diagnosed)
* 24 blood measurements. Today, two of them:
  * `fGSH` — free **glutathione**, the body's main antioxidant (µM)
  * `GSSG` — *oxidized* glutathione: glutathione that has been "used up" fighting oxidation (µM)

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split

autism = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/autism.csv")
autism["Group"].value_counts()

In [ ]:
kids = autism[autism["Group"] != "SIB"]          # today: ASD vs. NEU
X = kids[["fGSH", "GSSG"]]                       # features: two blood measurements
y = kids["Group"]                                # label

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7)
print(len(X_train), "training children |", len(X_test), "test children")

In [ ]:
COLORS = {"ASD": "#eb6834", "NEU": "#2a78d6"}
SYMBOLS = {"ASD": "triangle-up", "NEU": "circle"}

fig = px.scatter(X_train.assign(Group=y_train), x="fGSH", y="GSSG", color="Group", symbol="Group",
                 color_discrete_map=COLORS, symbol_map=SYMBOLS, title="111 Training Children")
fig.show()

> 💬 **Question:** Where do the ASD children sit? The NEU children?

<details><summary>Answer</summary>

* **ASD:** more *used-up* glutathione (higher `GSSG`), less *free* glutathione (lower `fGSH`) — upper left
* **NEU:** lower right
* A lot of overlap in the middle
</details>

---
## 2 · One Question: a Tree With One Split (SLO 07A) · ~10 min

> 📄 **Handout 1 — One line.** Draw **one** straight line, either **vertical** or **horizontal**, that best separates ASD from NEU. Count the children on the wrong side. 🗣️ Compare with a neighbor: who made fewer mistakes?

The computer plays the same game. It tries **every** possible cut on **every** feature and keeps the best one.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

tree1 = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_train, y_train)

plt.figure(figsize=(7, 3.5))
plot_tree(tree1, feature_names=X.columns, class_names=tree1.classes_, filled=True)
plt.show()

**Reading a box (a *node*):**

| line | means |
|---|---|
| `GSSG <= 0.159` | the **question**: true → go left, false → go right |
| `samples` | children who reach this box |
| `value` | how many of them are [ASD, NEU] |
| `class` | the majority, which is the **prediction** here |
| `gini` | how **mixed** the box is: 0 = all one group, 0.5 = half and half |

The tree picks the question that leaves the two boxes **least mixed**.

<details><summary>Answer: the tree's line</summary>

* One horizontal line at **GSSG = 0.159**: below → NEU, above → ASD
* Mistakes: 5 ASD below the line + 19 NEU above it = **24 of 111** → 78.4% right
* Did your line beat the computer's?
</details>

In [ ]:
print("training accuracy:", round(accuracy_score(y_train, tree1.predict(X_train)), 3))
print("test accuracy:    ", round(accuracy_score(y_test, tree1.predict(X_test)), 3))

---
## 3 · Two More Questions: Depth 2 · ~7 min

Each box can ask its **own** next question. `max_depth=2`: at most 2 questions per child.

In [ ]:
tree2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)

plt.figure(figsize=(11, 5))
plot_tree(tree2, feature_names=X.columns, class_names=tree2.classes_, filled=True)
plt.show()

> 📄 **Handout 2 — Follow the tree.** Send three test children down the tree. What does it predict for each? Was it right? 🗣️ Compare with a neighbor.

In [ ]:
three = X_test.loc[[49, 84, 40]].assign(true=y_test, predicted=tree2.predict(X_test.loc[[49, 84, 40]]))
three

<details><summary>Answer</summary>

* **Child 1** (fGSH 1.931, GSSG 0.209): GSSG > 0.159 → fGSH ≤ 2.147 → **ASD** ✓
* **Child 2** (2.537, 0.288): GSSG > 0.159 → fGSH > 2.147 → **NEU** ✓
* **Child 3** (1.318, 0.105): GSSG ≤ 0.159 → fGSH ≤ 1.672 → **NEU** ✗, an ASD child. Its box held 2 ASD and 3 NEU: nearly a coin flip
</details>

Every question cuts the plot with a vertical or horizontal line, so the tree's **map** is made of **rectangles**: one rectangle per final box (a **leaf**).

In [ ]:
# @title Helper: draw the map of one or more trees (double-click to see the code)
import numpy as np

def plot_maps(models):
    """models: {title: unfitted classifier}. Each is fit on the training children; the background is its prediction."""
    xx, yy = np.meshgrid(np.linspace(1.0, 5.6, 400), np.linspace(0.04, 0.68, 300))
    grid = pd.DataFrame({"fGSH": xx.ravel(), "GSSG": yy.ravel()})
    fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4), sharey=True, squeeze=False)
    for ax, (title, model) in zip(axes[0], models.items()):
        model.fit(X_train, y_train)
        z = (model.predict(grid) == "NEU").astype(int).reshape(xx.shape)
        ax.contourf(xx, yy, z, levels=[-0.5, 0.5, 1.5], colors=[COLORS["ASD"], COLORS["NEU"]], alpha=0.22)
        for group, marker in [("ASD", "^"), ("NEU", "o")]:
            pts = X_train[y_train == group]
            ax.scatter(pts["fGSH"], pts["GSSG"], s=18, c=COLORS[group], marker=marker, label=group,
                       edgecolors="white", linewidths=0.4)
        ax.set_title(f"{title}  ({model.get_n_leaves()} leaves)")
        ax.set_xlabel("fGSH (µM)")
    axes[0][0].set_ylabel("GSSG (µM)")
    axes[0][0].legend(loc="upper right")
    plt.show()

In [ ]:
plot_maps({"depth 1": DecisionTreeClassifier(max_depth=1, random_state=0),
           "depth 2": DecisionTreeClassifier(max_depth=2, random_state=0)})

---
## 4 · Too Many Questions: Overfitting (SLO 07A) · ~10 min

> 🎯 **Predict First:** With no limit on depth, the tree keeps asking until every leaf is **pure** (one group only). What will it score on the training children? On the test children?

In [ ]:
tree_full = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)      # no max_depth: grow until pure

print("depth:", tree_full.get_depth(), "| leaves:", tree_full.get_n_leaves())
print("training accuracy:", round(accuracy_score(y_train, tree_full.predict(X_train)), 3))
print("test accuracy:    ", round(accuracy_score(y_test, tree_full.predict(X_test)), 3))

In [ ]:
plt.figure(figsize=(20, 9))
plot_tree(tree_full, feature_names=X.columns, class_names=tree_full.classes_, filled=True, fontsize=6)
plt.show()

<details><summary>Answer</summary>

* **100%** on the training children; every leaf is pure
* **72.9%** on the test children: worse than one single question (79.2%)
* 23 leaves for 111 children: many leaves hold just **one or two** children
</details>

In [ ]:
plot_maps({"depth 1": DecisionTreeClassifier(max_depth=1, random_state=0),
           "depth 2": DecisionTreeClassifier(max_depth=2, random_state=0),
           "no limit": DecisionTreeClassifier(random_state=0)})

> 📄 **Handout 3 — Three maps.** Circle a rectangle in the "no limit" map that exists for a single child. Under each map, write *too simple*, *about right* or *too wiggly*.

<details><summary>Answer</summary>

* **Depth 1 — too simple:** one line can't use `fGSH` at all → **underfitting**
* **No limit — too wiggly:** thin rectangles around single children follow the *noise* → **overfitting**
* **Depth 2 — about right**
</details>

Which depth is best? Score every depth on both sets.

In [ ]:
scores = []
for depth in range(1, 11):
    model = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    scores.append({"max_depth": depth,
                   "training children": accuracy_score(y_train, model.predict(X_train)),
                   "test children": accuracy_score(y_test, model.predict(X_test))})
scores = pd.DataFrame(scores).melt(id_vars="max_depth", var_name="scored on", value_name="accuracy")

fig = px.line(scores, x="max_depth", y="accuracy", color="scored on", line_dash="scored on", markers=True,
              title="Decision Tree Accuracy vs. Depth")
fig.show()

> 📄 **Handout 4 — Which depth?** Circle the best depth for the test children. Label the underfitting zone and the overfitting zone.

<details><summary>Answer</summary>

* **Best on test: depth 2 or 3** (87.5%)
* **Depth 1:** poor on both → **underfitting**
* **Depth 4 and up:** training keeps climbing to 100%, test drops to 72.9% → **overfitting**
* From depth 7 on, nothing changes: every leaf is already pure
</details>

| | on data it learned from | on new data |
|---|---|---|
| **Overfitting** — too flexible | great | worse |
| **Underfitting** — too simple | poor | poor |
| **Good fit** | good | good |

Same table as kNN. Its knob was `k`, and a **small** `k` overfit. The tree's knob is `max_depth`, and a **large** depth overfits.

---
## 5 · Keeping Trees Small: Pruning · ~10 min

The full tree's problem: 23 leaves, many of them built for **one or two** children. Two ways to stop that:

| | idea | knobs in scikit-learn |
|---|---|---|
| **Pre-pruning** | **stop early**: refuse some questions while the tree grows | `max_depth` · `min_samples_leaf` · `min_samples_split` |
| **Post-pruning** | grow the full tree, then **cut back** the branches that help least | `ccp_alpha` |

**Pre-pruning: three stop rules**

| knob | the rule | to prune *more* |
|---|---|---|
| `max_depth=2` | at most 2 questions per child | make it **smaller** |
| `min_samples_leaf=10` | every leaf must hold at least 10 children. A question that would leave fewer on one side is not allowed | make it **bigger** |
| `min_samples_split=20` | a box with fewer than 20 children is not asked another question | make it **bigger** |

We used `max_depth` already. Now `min_samples_leaf`:

In [ ]:
tree_leaf = DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(X_train, y_train)   # every leaf: 10+ children

print("leaves:", tree_leaf.get_n_leaves())
print("training accuracy:", round(accuracy_score(y_train, tree_leaf.predict(X_train)), 3))
print("test accuracy:    ", round(accuracy_score(y_test, tree_leaf.predict(X_test)), 3))

In [ ]:
plot_maps({"no limit": DecisionTreeClassifier(random_state=0),
           "min_samples_leaf=10": DecisionTreeClassifier(min_samples_leaf=10, random_state=0)})

* **23 → 7 leaves.** No leaf may hold fewer than 10 children, so no rectangle can exist for a single child
* Training accuracy: 100% → 89.2%. Test accuracy: 72.9% → **87.5%**
* Some neighboring leaves give the same answer, so the map is the depth-2 map again

**Post-pruning: `ccp_alpha`** (*cost-complexity pruning*)

* Grow the **full** tree first. Then every leaf has a **price**: `ccp_alpha`
* A branch stays only if it makes the tree **less mixed** by more than the price of the leaves it adds. Otherwise it is cut, and the box above it becomes a leaf
* *Mixed* is measured with `gini`, averaged over the leaves (bigger leaves count more)
* `ccp_alpha=0`: leaves are free → the full tree. **Bigger `ccp_alpha` → more is cut → smaller tree**

Our full tree, at a price of **0.02** per leaf:

| branch | leaves it adds | mix it removes | per leaf | at `ccp_alpha=0.02` |
|---|---|---|---|---|
| the second question, `fGSH <= 2.147` | 1 | 0.143 | 0.143 | worth it → **stays** |
| everything below that | 20 | 0.187 in total | at most 0.013 | too expensive → **cut** |

In [ ]:
tree_ccp = DecisionTreeClassifier(ccp_alpha=0.02, random_state=0).fit(X_train, y_train)   # each leaf costs 0.02

print("leaves:", tree_ccp.get_n_leaves())
print("training accuracy:", round(accuracy_score(y_train, tree_ccp.predict(X_train)), 3))
print("test accuracy:    ", round(accuracy_score(y_test, tree_ccp.predict(X_test)), 3))

* **23 → 3 leaves**, test accuracy 87.5%: two questions survive, `GSSG <= 0.159` and `fGSH <= 2.147`

`ccp_alpha` is a knob too. Try a few prices:

In [ ]:
rows = []
for alpha in [0, 0.01, 0.02, 0.2]:
    model = DecisionTreeClassifier(ccp_alpha=alpha, random_state=0).fit(X_train, y_train)
    rows.append({"ccp_alpha": alpha,
                 "leaves": model.get_n_leaves(),
                 "training accuracy": accuracy_score(y_train, model.predict(X_train)),
                 "test accuracy": accuracy_score(y_test, model.predict(X_test))})
pd.DataFrame(rows).round(3)

* **`ccp_alpha=0`:** nothing is cut, 23 leaves → **overfitting**
* **`ccp_alpha=0.2`:** so expensive that even the first question is cut. One leaf: everyone gets the same label → **underfitting**
* **In between:** 3 leaves, the best test accuracy
* Every pruning knob has these two ends, just like `k` in kNN

**The three knobs, side by side:**

| | `max_depth` | `min_samples_leaf` | `ccp_alpha` |
|---|---|---|---|
| **kind** | pre-pruning | pre-pruning | post-pruning |
| **the rule** | at most this many questions per child | every leaf holds at least this many children | every leaf costs this much; branches not worth it are cut |
| **default** | no limit | 1 | 0 |
| **to prune more** | make it **smaller** | make it **bigger** | make it **bigger** |
| **overfitting end** | no limit | 1 | 0 |
| **underfitting end** | 1 | very large | very large |
| **today's value** | 2 → 4 leaves | 10 → 7 leaves | 0.02 → 3 leaves |
| **test accuracy** | 87.5% | 87.5% | 87.5% |

But *which* depth, which `min_samples_leaf`, which `ccp_alpha`? We just chose them by peeking at the test set. **Wednesday:** a fair way to choose.

**Decision trees, in short:**

| 👍 | 👎 |
|---|---|
| you can **read** them: a list of yes/no questions | **overfit** easily if left to grow |
| no scaling needed: each question looks at one column, so units don't matter | **unstable**: a few different children can grow a different tree |

---
## 6 · A Rule for Sorting Children · ~3 min

Our depth-2 tree, printed on a card:

> **GSSG above 0.159 *and* fGSH at most 2.147 → autism. Otherwise → no autism.**

> "Rule-based “algorithms” and “clusters of attributes” have always served to draw boundaries between kinds of people and things."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** Unlike most models, this one is a rule anyone can read. If a clinic used this card on a 3-year-old, what would you still want to know?

<details><summary>Answer</summary>

* **Whose blood?** 159 children from one study in Arkansas. Would the cutoffs hold elsewhere?
* **How often wrong?** About 1 test child in 8, and right at the line a child is a near coin flip (child 3)
* **A sharp border on a smooth measure:** GSSG 0.158 and 0.160 get different labels
* Readable makes a rule easier to question. It doesn't make it right
</details>

**Wednesday:** how to choose depth (and other knobs) *without* touching the test set — **cross-validation** and **grid search** *(SLOs 07B, 07C)*.